In [3]:
# Task 1: LangChain Setup & Core Concepts (Groq)
# pip install langchain langchain-groq langchain-core
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


load_dotenv()
# Setup Groq LLM
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

# LCEL chain: prompt | llm | parser
prompt = ChatPromptTemplate.from_template("Answer in 2-3 sentences: {question}")
parser = StrOutputParser()
chain = prompt | llm | parser

# Test invoke
result = chain.invoke({"question": "What is LangChain Expression Language?"})
print("INVOKE RESULT:\n", result)

# Test stream
print("\nSTREAM RESULT:")
for chunk in chain.stream({"question": "Why use pipe syntax?"}):
    print(chunk, end="", flush=True)
print()

INVOKE RESULT:
 LangChain Expression Language (LCEL) is a concise, declarative syntax for building LangChain pipelines that lets developers chain together prompts, LLM calls, tools, and other components without writing extensive boilerplate code. By expressing workflows as readable expressions, LCEL simplifies the creation, composition, and debugging of complex LLM‑driven applications.

STREAM RESULT:
Pipe syntax lets you chain operations in a clear, left‑to‑right flow, making code easier to read and reason about because each step receives the output of the previous one. It also encourages composability, letting you build complex behavior from simple, reusable functions without deeply nested calls or temporary variables. In many environments (e.g., Unix shells, functional languages) the pipe automatically handles data passing, reducing boilerplate and potential errors.


In [ ]:
# Task 2: Define & Register Tools
import json
from langchain_core.tools import tool

# Create products.json for the real data source tool
products_data = {"products": [
    {"name": "laptop", "price": 999, "category": "electronics", "stock": 15},
    {"name": "phone", "price": 699, "category": "electronics", "stock": 30},
    {"name": "tablet", "price": 449, "category": "electronics", "stock": 20},
    {"name": "headphones", "price": 149, "category": "accessories", "stock": 50},
    {"name": "smartwatch", "price": 299, "category": "wearables", "stock": 25}
]}
with open("products.json", "w") as f:
    json.dump(products_data, f, indent=2)

# Tool 1: Calculator (reused from Day 1)
@tool
def calculator(operation: str, a: float, b: float) -> str:
    """Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics."""
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"{a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

# Tool 2: Weather (reused from Day 1)
@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate conditions for a specific city. Returns fake stub data for testing."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15, "berlin": 20}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

# Tool 3: Product Lookup (NEW - reads real JSON data)
@tool
def product_lookup(product_name: str) -> str:
    """Looks up product information including price, category, and stock availability from the local product database. Use this when the user asks about product prices, availability, or wants to compare products."""
    with open("products.json", "r") as f:
        data = json.load(f)
    for p in data["products"]:
        if product_name.lower() in p["name"].lower():
            return f"Product: {p['name']}, Price: ${p['price']}, Category: {p['category']}, Stock: {p['stock']} units"
    return f"Product '{product_name}' not found in database"

# Register tools
tools = [calculator, get_weather, product_lookup]
# Test each tool individually
print("=== TOOL TESTS ===")
print("calculator:", calculator.invoke({"operation": "add", "a": 10, "b": 5}))
print("get_weather:", get_weather.invoke({"city": "Tokyo"}))
print("product_lookup:", product_lookup.invoke({"product_name": "laptop"}))

# Show tool schemas (what the LLM sees)
print("\n=== TOOL SCHEMAS ===")
for t in tools:
    print(f"\n{t.name}: {t.description}")
    print(f"  Schema: {t.args_schema.model_json_schema()}")

=== TOOL TESTS ===
calculator: 10.0 + 5.0 = 15.0
get_weather: Weather in Tokyo: 22°C
product_lookup: Product: laptop, Price: $999, Category: electronics, Stock: 15 units

=== TOOL SCHEMAS ===

calculator: Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics.
  Schema: {'description': 'Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics.', 'properties': {'operation': {'title': 'Operation', 'type': 'string'}, 'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['operation', 'a', 'b'], 'title': 'calculator', 'type': 'object'}

get_weather: Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate co

In [ ]:
# Task 3: Build an Agent
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain.agents import create_agent
import json

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """Performs basic arithmetic (add, subtract) on two numbers. Use this when the user asks for math calculations like adding, subtracting, or any arithmetic operation. Do not use for comparisons or statistics."""
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"{a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use this when the user asks about weather, temperature, or climate conditions for a specific city. Returns fake stub data for testing."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15, "berlin": 20}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product information including price, category, and stock availability from the local product database. Use this when the user asks about product prices, availability, or wants to compare products."""
    with open("products.json", "r") as f:
        data = json.load(f)
    for p in data["products"]:
        if product_name.lower() in p["name"].lower():
            return f"Product: {p['name']}, Price: ${p['price']}, Category: {p['category']}, Stock: {p['stock']} units"
    return f"Product '{product_name}' not found in database"

tools = [calculator, get_weather, product_lookup]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the tools available to answer questions."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_agent(llm, tools, prompt)

result = agent.invoke({"input": "What's the weather in Tokyo and Paris? Which is warmer?", "chat_history": []})
print("FINAL ANSWER:", result)

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\muham\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain\agents\__init__.py)

In [13]:
import langchain.agents
print(dir(langchain.agents))

['AgentState', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_subagent_transformer', 'create_agent', 'factory', 'middleware', 'structured_output']
